# module-composition composite — cx11: nn.Sequential body lives inside an nn.Module subclass

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `module-composition`, `nn-module-subclass`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
from einops.layers.torch import Rearrange

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "module-composition"
DD_ATOM_IDS = ["module-composition", "nn-module-subclass"]
DD_SUBTOPICS = ["PyTorch: Module composition", "PyTorch: nn.Module subclassing"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Sometimes you need a custom Module — for a non-Sequential forward (e.g. taking BOTH `(x, y)` as input, or returning a tuple), or to attach helper methods (`.encode(x)`, `.generate(n)`). But the bulk of the forward is still a STACK of layers. The idiomatic pattern is to:

- Subclass `nn.Module`.
- Hold the layer stack as a NAMED ATTRIBUTE that is itself an `nn.Sequential` (or a list of children wrapped in `nn.Sequential`).
- Reference the stack inside `forward` as `self.layers(x)`.

**Why both atoms together.** The `nn.Module` subclass is the API (you get `.parameters()`, `.to(device)`, `.train()/.eval()`, and a place for helper methods). The `nn.Sequential` attribute is the COMPOSITION (you get a clean linear stack without writing `forward` for each layer).

**Anatomy.**
```python
class Generator(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()                       # nn-module-subclass.
        self.project = nn.Linear(latent_dim, 256 * 4 * 4)
        self.layers = nn.Sequential(             # module-composition: layer stack as a CHILD.
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 1, 4, 2, 1),
            nn.Tanh(),
        )
    def forward(self, z):
        h = self.project(z).view(-1, 256, 4, 4)
        return self.layers(h)
    def generate(self, n):                       # helper only possible because we subclassed.
        return self(t.randn(n, self.project.in_features))
```

### Composite Exercise — nn.Sequential body lives inside an nn.Module subclass

**Atoms exercised together**: `module-composition`, `nn-module-subclass`

Implement `cx11_make_generator_class()` — return a CLASS `Generator(nn.Module)` implementing a tiny GAN generator with a Linear projection + a Sequential stack.

Required structure:
1. `__init__(self, latent_dim)`:
   - `super().__init__()` first.
   - `self.project = nn.Linear(latent_dim, 64 * 4 * 4)`  # Linear projection.
   - `self.layers = nn.Sequential(`  # the layer stack as a named child.
       `nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1, bias=False),`
       `nn.BatchNorm2d(32),`
       `nn.ReLU(inplace=True),`
       `nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),`
       `nn.Tanh(),`
     `)`  # (B, 64, 4, 4) -> (B, 32, 8, 8) -> (B, 1, 16, 16).
2. `forward(self, z)`:
   - z is `(B, latent_dim)`.
   - `h = self.project(z)` -> `(B, 64*4*4)`.
   - reshape to `(B, 64, 4, 4)`.
   - return `self.layers(h)` -> `(B, 1, 16, 16)`.
3. `generate(self, n)` — convenience method:
   - Sample `z = t.randn(n, self.project.in_features)`.
   - Return `self(z)` (uses `__call__`, not `.forward`).

Test checks:
- `cx11_make_generator_class()` returns a class subclassing `nn.Module`.
- Instance has both `project` (Linear) and `layers` (Sequential) as named children.
- `forward(z)` produces `(B, 1, 16, 16)`.
- `generate(n)` produces `(n, 1, 16, 16)` and uses `__call__`.
- All parameters from BOTH `project` and `layers` show up in `.parameters()`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx11_make_generator_class():
    """Return the Generator class."""
    raise NotImplementedError

def _test_cx11():
    Generator = cx11_make_generator_class()
    assert isinstance(Generator, type) and issubclass(Generator, nn.Module)

    # Case A: instantiation + super().__init__() proof.
    t.manual_seed(0)
    gen = Generator(latent_dim=10)
    params = list(gen.parameters())
    assert len(params) > 2, (
        f'too few params ({len(params)}) — did you forget super().__init__()?'
    )

    # Case B: named children — project AND layers.
    named = dict(gen.named_children())
    assert 'project' in named and isinstance(named['project'], nn.Linear), (
        f"expected 'project' Linear child; got {list(named)}"
    )
    assert 'layers' in named and isinstance(named['layers'], nn.Sequential), (
        f"expected 'layers' Sequential child; got {list(named)}"
    )
    assert named['project'].in_features == 10
    assert named['project'].out_features == 64 * 4 * 4

    # Case C: forward shape.
    z = t.randn(3, 10)
    out = gen(z)
    assert out.shape == (3, 1, 16, 16), f'expected (3, 1, 16, 16); got {tuple(out.shape)}'

    # Case D: layers stack has the right composition (5 layers).
    layer_seq = list(named['layers'].children())
    assert len(layer_seq) == 5, f'layers should have 5 inner modules; got {len(layer_seq)}'
    assert isinstance(layer_seq[0], nn.ConvTranspose2d)
    assert isinstance(layer_seq[1], nn.BatchNorm2d)
    assert isinstance(layer_seq[2], nn.ReLU)
    assert isinstance(layer_seq[3], nn.ConvTranspose2d)
    assert isinstance(layer_seq[4], nn.Tanh)

    # Case E: generate(n) shape and that it uses __call__.
    t.manual_seed(1)
    samples = gen.generate(5)
    assert samples.shape == (5, 1, 16, 16), f'expected (5, 1, 16, 16); got {tuple(samples.shape)}'

    # Case F: parameters from BOTH children are recursively collected.
    # project: Linear -> 2 params (weight + bias).
    # layers: ConvT(bias=F)+BN(2) + ConvT(bias=T)+Tanh(0) = 1+2+2 = 5 params.
    # total = 7.
    assert len(params) == 7, f'expected 7 total params (2 project + 5 layers); got {len(params)}'
    _dd_passed.add('cx11')

_test_cx11()

<details><summary>Show solution — cx11</summary>

```python
def cx11_make_generator_class():
    class Generator(nn.Module):
        def __init__(self, latent_dim):
            # Atom B (nn-module-subclass): wire up the module registry first.
            super().__init__()
            self.project = nn.Linear(latent_dim, 64 * 4 * 4)
            # Atom A (module-composition): nn.Sequential held as a named child.
            self.layers = nn.Sequential(
                nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
                nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),
                nn.Tanh(),
            )

        def forward(self, z):
            h = self.project(z)
            h = h.view(-1, 64, 4, 4)
            return self.layers(h)

        def generate(self, n):
            z = t.randn(n, self.project.in_features)
            return self(z)  # __call__, not .forward — runs hooks.

    return Generator
```

Notice `self.layers` is a single attribute holding the WHOLE Sequential — child-of-child registration works recursively, so `gen.parameters()` finds the ConvT weights inside `layers` without any extra wiring. The `generate(n)` helper is the payoff: it's not possible to attach helper methods to a bare `nn.Sequential`, so when you want them, you subclass.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx11'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx11',
        'subtopics': ["PyTorch: Module composition", "PyTorch: nn.Module subclassing"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()